In [5]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
viz_bidirectional_4244.py
-------------------------
4.2.4.4 双方向（OH→FP と FP→OH）比較の可視化まとめ。

入力（既定パスに無い場合は --base で指定）:
  <base>/paper_4.2.4.4_OHFP/bidirectional_summary/
    ├─ Table_4.2.4.4_bidirectional_joined.csv
    └─ Table_4.2.4.4_bidirectional_correlations.csv

出力:
  <base>/paper_4.2.4.4_OHFP/bidirectional_summary/visuals/
    ├─ Fig_4.2.4.4_corr_heatmap.png/pdf
    ├─ Fig_4.2.4.4_scatter_{X}_vs_{Y}.png/pdf      （主要ペア）
    ├─ Fig_4.2.4.4_scatter_labeled_{X}_vs_{Y}.png/pdf（mode/indexラベル注釈）
    └─ Fig_4.2.4.4_pairs_grid_{X}_vs_{Y}.png/pdf   （条件ごとグリッド）

実行例:
  python viz_bidirectional_4244.py
  python viz_bidirectional_4244.py --base /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/paper_4.2.4.4_OHFP_20251022
"""

from __future__ import annotations
import argparse, os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # GUIなし環境向け
import matplotlib.pyplot as plt

DEFAULT_BASE = Path("/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/paper_4.2.4.4_OHFP_20251022")

def load_tables(base: Path):
    src_dir = base / "paper_4.2.4.4_OHFP_20251022" / "bidirectional_summary"
    joined = src_dir / "Table_4.2.4.4_bidirectional_joined.csv"
    corrs  = src_dir / "Table_4.2.4.4_bidirectional_correlations.csv"
    if not joined.exists():
        raise FileNotFoundError(f"joined が見つかりません: {joined}")
    if not corrs.exists():
        raise FileNotFoundError(f"correlations が見つかりません: {corrs}")
    dfj = pd.read_csv(joined)
    dfc = pd.read_csv(corrs)
    return dfj, dfc, src_dir

def ensure_outdir(src_dir: Path):
    out = src_dir / "visuals"
    out.mkdir(parents=True, exist_ok=True)
    return out

def corr_heatmap_from_table(dfc: pd.DataFrame, outdir: Path):
    # correlations.csv -> ヒートマップ（Pearson と Spearman を2枚）
    for stat_col, title_suffix in [("pearson_r","(Pearson r)"), ("spearman_rho","(Spearman ρ)")]:
        # 行: forward_metric, 列: reverse_metric
        fms = list(dict.fromkeys(dfc["forward_metric"].astype(str)))
        rms = list(dict.fromkeys(dfc["reverse_metric"].astype(str)))
        M = np.full((len(fms), len(rms)), np.nan, dtype=float)
        for i, fm in enumerate(fms):
            for j, rm in enumerate(rms):
                sub = dfc[(dfc["forward_metric"]==fm) & (dfc["reverse_metric"]==rm)]
                if len(sub):
                    M[i,j] = float(sub.iloc[0][stat_col])

        fig, ax = plt.subplots(figsize=(1.2*len(rms)+2, 1.0*len(fms)+2), dpi=300)
        im = ax.imshow(M, aspect="auto", interpolation="nearest")
        ax.set_xticks(np.arange(len(rms))); ax.set_xticklabels(rms, rotation=35, ha="right")
        ax.set_yticks(np.arange(len(fms))); ax.set_yticklabels(fms)
        ax.set_title(f"Fig 4.2.4.4: Forward vs Reverse correlations {title_suffix}")
        ax.grid(False)
        cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.set_label("Correlation")
        # 数値を上に軽く入れる（NaNはスキップ）
        for i in range(len(fms)):
            for j in range(len(rms)):
                if not np.isnan(M[i,j]):
                    ax.text(j, i, f"{M[i,j]:.2f}", ha="center", va="center", fontsize=8)
        fig.tight_layout()
        fig.savefig(outdir / f"Fig_4.2.4.4_corr_heatmap_{stat_col}.png", bbox_inches="tight")
        fig.savefig(outdir / f"Fig_4.2.4.4_corr_heatmap_{stat_col}.pdf", bbox_inches="tight")
        plt.close(fig)

def scatter_with_fit(df, xcol, ycol, outdir: Path, label_points=False, grid_by_condition=False):
    # 散布 + 近似直線
    sub = df[[xcol, ycol, "mode", "index"]].copy()
    x = pd.to_numeric(sub[xcol], errors="coerce")
    y = pd.to_numeric(sub[ycol], errors="coerce")
    m = x.notna() & y.notna()
    if m.sum() < 2:
        return
    x, y, sub = x[m].values, y[m].values, sub[m]

    if grid_by_condition:
        # mode×index の小 multiples
        keys = [(r.mode, r.index) for r in sub.itertuples()]
        uniq = sorted(set(keys))
        n = len(uniq)
        cols = min(3, max(1, int(np.ceil(np.sqrt(n)))))
        rows = int(np.ceil(n / cols))
        fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 3.8*rows), dpi=300)
        if rows*cols == 1:
            axes = np.array([[axes]])
        axes = axes.reshape(rows, cols)
        k = 0
        for i in range(rows):
            for j in range(cols):
                ax = axes[i,j]
                if k >= n:
                    ax.axis("off")
                    continue
                mkey = uniq[k]
                sel = (sub["mode"]==mkey[0]) & (sub["index"]==mkey[1])
                xx = pd.to_numeric(sub.loc[sel, xcol], errors="coerce")
                yy = pd.to_numeric(sub.loc[sel, ycol], errors="coerce")
                mm = xx.notna() & yy.notna()
                if mm.sum() >= 2:
                    ax.scatter(xx[mm].values, yy[mm].values, alpha=0.9)
                    # fit
                    coef = np.polyfit(xx[mm].values, yy[mm].values, 1)
                    xs = np.linspace(float(np.min(xx[mm].values)), float(np.max(xx[mm].values)), 100)
                    ys = np.polyval(coef, xs)
                    ax.plot(xs, ys)
                ax.set_title(f"{mkey[0]} × {mkey[1]}")
                ax.set_xlabel(xcol); ax.set_ylabel(ycol)
                ax.grid(True, linestyle="--", linewidth=0.4, alpha=0.6)
                k += 1
        fig.suptitle(f"Fig 4.2.4.4: {xcol} vs {ycol} — by (mode,index)")
        fig.tight_layout(rect=[0,0,1,0.97])
        fig.savefig(outdir / f"Fig_4.2.4.4_pairs_grid_{xcol}_vs_{ycol}.png", bbox_inches="tight")
        fig.savefig(outdir / f"Fig_4.2.4.4_pairs_grid_{xcol}_vs_{ycol}.pdf", bbox_inches="tight")
        plt.close(fig)
        return

    fig, ax = plt.subplots(figsize=(6,4), dpi=300)
    ax.scatter(x, y, alpha=0.9)
    # 近似直線
    coef = np.polyfit(x, y, 1)
    xs = np.linspace(float(np.min(x)), float(np.max(x)), 100)
    ys = np.polyval(coef, xs)
    ax.plot(xs, ys)
    ax.set_xlabel(xcol); ax.set_ylabel(ycol)
    ttl = f"Fig 4.2.4.4: {xcol} vs {ycol}"
    ax.set_title(ttl)
    ax.grid(True, linestyle="--", linewidth=0.4, alpha=0.6)

    if label_points:
        for r in sub.itertuples():
            ax.text(getattr(r, xcol), getattr(r, ycol), f"{r.mode}-{r.index}",
                    ha="left", va="bottom", fontsize=8)

    fig.tight_layout()
    base = f"Fig_4.2.4.4_scatter_{'labeled_' if label_points else ''}{xcol}_vs_{ycol}"
    fig.savefig(outdir / (base + ".png"), bbox_inches="tight")
    fig.savefig(outdir / (base + ".pdf"), bbox_inches="tight")
    plt.close(fig)

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--base", type=str, default=str(DEFAULT_BASE),
                    help="プロジェクトのベースフォルダ（既定は提示いただいたパス）")
    args, _ = ap.parse_known_args()  # Jupyterの -f 回避
    base = Path(args.base).resolve()

    dfj, dfc, src_dir = load_tables(base)
    outdir = ensure_outdir(src_dir)

    # 1) 相関ヒートマップ（Pearson & Spearman）
    corr_heatmap_from_table(dfc, outdir)

    # 2) 主要ペアの散布図（全体 + ラベル付き + 条件グリッド）
    pairs = [
        ("ARI_mean", "pair_OH_major_same_rate"),
        ("ARI_mean", "pair_mean_cosine_similarity"),
        ("ARI_mean", "FPcluster_median_cohesive_ratio"),
        ("ARI_mean", "mean_OH_purity"),
        ("ARI_mean", "mean_OH_entropy"),
    ]
    for xcol, ycol in pairs:
        if xcol in dfj.columns and ycol in dfj.columns:
            scatter_with_fit(dfj, xcol, ycol, outdir, label_points=False, grid_by_condition=False)
            scatter_with_fit(dfj, xcol, ycol, outdir, label_points=True,  grid_by_condition=False)
            scatter_with_fit(dfj, xcol, ycol, outdir, label_points=False, grid_by_condition=True)

    print("\n✅ Done. Figures ->", outdir)

if __name__ == "__main__":
    main()


FileNotFoundError: joined が見つかりません: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/paper_4.2.4.4_OHFP_20251022/paper_4.2.4.4_OHFP_20251022/bidirectional_summary/Table_4.2.4.4_bidirectional_joined.csv

In [3]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
check_empty_files.py
--------------------
指定したディレクトリ（または既定の 4.2.4.4 関連フォルダ群）を再帰的に探索し、
0バイト（空）ファイルを検出・表示します。

実行例:
  python check_empty_files.py \
    --base /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/paper_4.2.4.4_OHFP_20251022
"""

from __future__ import annotations
import argparse
from pathlib import Path

DEFAULT_BASE = Path("/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No")

def find_empty_files(base: Path, exts: tuple[str, ...] = (".csv", ".txt", ".tsv")) -> list[Path]:
    """指定フォルダ以下の空ファイル（0バイト）を再帰探索"""
    empty = []
    for p in base.rglob("*"):
        if not p.is_file():
            continue
        if exts and not p.suffix.lower() in exts:
            continue
        try:
            size = p.stat().st_size
        except Exception:
            continue
        if size == 0:
            empty.append(p)
    return empty

def summarize_sizes(base: Path, top_n: int = 10):
    """サイズ上位/下位も参考までに出す"""
    files = [p for p in base.rglob("*") if p.is_file()]
    infos = [(p, p.stat().st_size) for p in files]
    infos.sort(key=lambda x: x[1])
    print(f"\n最小サイズファイル Top {top_n}:")
    for p, s in infos[:top_n]:
        print(f"  {s:>8,d} bytes — {p}")
    infos.sort(key=lambda x: x[1], reverse=True)
    print(f"\n最大サイズファイル Top {top_n}:")
    for p, s in infos[:top_n]:
        print(f"  {s:>8,d} bytes — {p}")

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--base", type=str, default=str(DEFAULT_BASE),
                    help="探索ベースディレクトリ（既定: cal_cluster_No）")
    ap.add_argument("--exts", type=str, default=".csv,.txt,.tsv",
                    help="探索対象拡張子（カンマ区切り）")
    args, _ = ap.parse_known_args()

    base = Path(args.base).resolve()
    if not base.exists():
        raise FileNotFoundError(f"指定ディレクトリが存在しません: {base}")

    exts = tuple(e.strip().lower() for e in args.exts.split(",") if e.strip())

    print(f"🔍 Searching under: {base}")
    empty_files = find_empty_files(base, exts=exts)

    if empty_files:
        print(f"\n⚠️ 空 (0バイト) ファイルが {len(empty_files)} 件見つかりました:")
        for p in empty_files:
            print("   -", p)
    else:
        print("\n✅ 空ファイルは見つかりませんでした。")

    # サイズ分布の概要も表示
    summarize_sizes(base)

if __name__ == "__main__":
    main()

🔍 Searching under: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No

⚠️ 空 (0バイト) ファイルが 1 件見つかりました:
   - /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/paper_exports_20251008_161116/untitled.txt

最小サイズファイル Top 10:
         0 bytes — /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/paper_exports_20251008_161116/untitled.txt
         4 bytes — /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/paper_4.2.4.4_OHFP/bidirectional_summary/Table_4.2.4.4_bidirectional_correlations.csv
         4 bytes — /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/paper_4.2.4.4_OHFP_20251022/paper_4.2.4.4_OHFP/bidirectional_summary/Table_4.2.4.4_bidirectional_correlations.csv
         7 bytes — /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/OHdata/hubert_DataGrp.csv
         7 bytes — /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/OHdata/ratkowsky_DataGrp_notDel.csv
         7 bytes — /Volumes/csbdeep11/_yasu-i/20250303_rebuiled

In [6]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
viz_4244_from_joined_autofix.py
--------------------------------
状況:
- これまで: joined(結合)CSV + correlations(相関)CSVを読み、可視化していた
- 今回: correlations.csv が空 (0バイト) で EmptyDataError

対応:
- correlations.csv が無い/空/壊れている場合、joined.csv から相関を自動再計算して保存し、続行
- Jupyterの -f を無視
- 最低限の図: 相関ヒートマップ（Pearson/Spearman）、主要散布図一式

想定ファイルの候補:
  1) <base>/paper_4.2.4.4_OHFP/bidirectional_summary/
       - Table_4.2.4.4_bidirectional_joined.csv
       - Table_4.2.4.4_bidirectional_correlations.csv
  2) <base>/paper_4.2.4.4_OHFP_2025xxxx/... （日付違い）も探索

必要に応じて --base / --joined / --corrs で明示指定してください。
"""

from __future__ import annotations
import argparse
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ---------- ユーティリティ ----------
def read_csv_safe(path: Path) -> pd.DataFrame | None:
    """空や壊れに強く、あればDataFrame / 読めなければ None を返す"""
    if not path or not path.exists() or path.stat().st_size == 0:
        return None
    for enc in ("utf-8", "utf-8-sig", "cp932"):
        try:
            df = pd.read_csv(path, encoding=enc)
            # 列が無い/0行もNG扱いにするなら以下で弾く
            if df is None or df.shape[1] == 0:
                return None
            return df
        except Exception:
            continue
    return None

def discover_default_base(base_hint: Path | None) -> Path | None:
    """ヒントが無ければよく使う既定パスを返す"""
    if base_hint and base_hint.exists():
        return base_hint
    # よく使う既定
    cands = [
        Path("/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No"),
    ]
    for c in cands:
        if c.exists():
            return c
    return None

def find_joined_and_corrs(base: Path, joined_arg: Path | None, corrs_arg: Path | None):
    """よくある場所を総当たりして joined/corrs を見つける"""
    # 1) 明示指定があればそれを最優先
    if joined_arg and joined_arg.exists():
        joined = joined_arg
    else:
        joined = None

    if corrs_arg and corrs_arg.exists():
        corrs = corrs_arg
    else:
        corrs = None

    # 2) 既定のサブディレクトリ群で探索
    subdirs = [
        base / "paper_4.2.4.4_OHFP" / "bidirectional_summary",
        base / "paper_4.2.4.4_OHFP_20251022" / "paper_4.2.4.4_OHFP" / "bidirectional_summary",
        base / "paper_4.2.4.4_OHFP_20251016_comp" / "paper_4.2.4.4_OHFP" / "bidirectional_summary",
    ]
    joined_names = [
        "Table_4.2.4.4_bidirectional_joined.csv",
        "Table_4.2.4.4_byVariant_joined_master.csv",  # 由来別マスターでも可視化できるように
    ]
    corrs_names = [
        "Table_4.2.4.4_bidirectional_correlations.csv",
    ]
    if joined is None:
        for sd in subdirs:
            for nm in joined_names:
                p = sd / nm
                if p.exists() and p.stat().st_size > 0:
                    joined = p; break
            if joined is not None:
                break
    if corrs is None:
        for sd in subdirs:
            for nm in corrs_names:
                p = sd / nm
                # 相関は空の可能性があるので存在チェックのみ
                if p.exists():
                    corrs = p; break
            if corrs is not None:
                break
    return joined, corrs

def compute_correlations(df: pd.DataFrame) -> pd.DataFrame:
    """Pearson & Spearman を両方出す（数値列のみ）"""
    num = df.select_dtypes(include=[np.number]).copy()
    # 欠損が多い列を落としたいならここで閾値処理も可
    pear = num.corr(method="pearson")
    spear = num.corr(method="spearman")
    # ロングにして2種類の相関を1表にまとめる
    r1 = pear.stack().rename("pearson").reset_index().rename(columns={"level_0":"X","level_1":"Y"})
    r2 = spear.stack().rename("spearman").reset_index().rename(columns={"level_0":"X","level_1":"Y"})
    out = r1.merge(r2, on=["X","Y"], how="outer")
    return out

def save_corr_heatmap(M: pd.DataFrame, metric: str, out_png: Path, out_pdf: Path, title: str):
    """相関行列をヒートマップで保存（metric: 'pearson' or 'spearman'）"""
    # 対称行列を復元
    piv = M.pivot(index="X", columns="Y", values=metric)
    # 軸を同じ順番にそろえる
    cols = list(piv.columns)
    piv = piv.reindex(index=cols, columns=cols)
    fig, ax = plt.subplots(figsize=(0.45*len(cols)+3, 0.45*len(cols)+2), dpi=300)
    im = ax.imshow(piv.values, aspect="auto", interpolation="nearest")
    ax.set_xticks(np.arange(len(cols))); ax.set_xticklabels(cols, rotation=35, ha="right", fontsize=8)
    ax.set_yticks(np.arange(len(cols))); ax.set_yticklabels(cols, fontsize=8)
    ax.set_title(title)
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04); cbar.set_label(metric)
    # 数値も入れる（小さめ）
    for i in range(len(cols)):
        for j in range(len(cols)):
            v = piv.values[i,j]
            if np.isfinite(v):
                ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=6)
    fig.tight_layout()
    out_png.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_png, bbox_inches="tight")
    fig.savefig(out_pdf, bbox_inches="tight")
    plt.close(fig)

def scatter_pair(df: pd.DataFrame, x: str, y: str, out_png: Path, out_pdf: Path, title: str):
    """散布図 + 単回帰線"""
    if x not in df.columns or y not in df.columns:
        return
    cx = pd.to_numeric(df[x], errors="coerce")
    cy = pd.to_numeric(df[y], errors="coerce")
    m = cx.notna() & cy.notna()
    if m.sum() < 3:
        return
    xvals, yvals = cx[m].values, cy[m].values
    fig, ax = plt.subplots(figsize=(5.8,4.2), dpi=300)
    ax.scatter(xvals, yvals, alpha=0.9)
    coef = np.polyfit(xvals, yvals, 1)
    xs = np.linspace(float(np.min(xvals)), float(np.max(xvals)), 100)
    ys = np.polyval(coef, xs)
    ax.plot(xs, ys)
    ax.set_xlabel(x); ax.set_ylabel(y); ax.set_title(title)
    ax.grid(True, linestyle="--", linewidth=0.4, alpha=0.6)
    fig.tight_layout()
    out_png.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_png, bbox_inches="tight")
    fig.savefig(out_pdf, bbox_inches="tight")
    plt.close(fig)

# ---------- メイン ----------
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--base", type=str, default=None, help="プロジェクトのベース（例: /Volumes/.../cal_cluster_No）")
    ap.add_argument("--joined", type=str, default=None, help="結合テーブルのCSVパス（任意）")
    ap.add_argument("--corrs", type=str, default=None, help="相関テーブルのCSVパス（任意）")
    args, _ = ap.parse_known_args()  # Jupyter の -f を無視

    base = discover_default_base(Path(args.base) if args.base else None)
    if base is None:
        raise FileNotFoundError("base が見つかりません。--base で指定してください。")

    joined_arg = Path(args.joined) if args.joined else None
    corrs_arg  = Path(args.corrs)  if args.corrs  else None
    joined, corrs = find_joined_and_corrs(base, joined_arg, corrs_arg)

    if not joined:
        raise FileNotFoundError("joined（結合CSV）が見つかりません。--joined で明示指定してください。")

    # 読み込み
    dfj = read_csv_safe(joined)
    if dfj is None:
        raise RuntimeError(f"joined が空/破損: {joined}")

    # 相関テーブルの確保（無い/空なら再計算）
    need_recompute = False
    dfc = read_csv_safe(corrs) if corrs else None
    if dfc is None:
        need_recompute = True
    else:
        # 必要列が無い/空のときも再計算
        if not set(["X","Y"]).issubset(dfc.columns) or ("pearson" not in dfc.columns and "spearman" not in dfc.columns):
            need_recompute = True

    out_dir = (joined.parent)  # 既存の bidirectional_summary に出す
    corrs_out = out_dir / "Table_4.2.4.4_bidirectional_correlations.csv"

    if need_recompute:
        print("[INFO] correlations を joined から再計算します…")
        dfc = compute_correlations(dfj)
        dfc.to_csv(corrs_out, index=False, encoding="utf-8-sig")
        print(f"[SAVE] {corrs_out}")
    else:
        print(f"[OK] correlations 読み込み: {corrs}")

    # === 図出力 ===
    fig_dir = out_dir / "figs_from_joined"
    fig_dir.mkdir(parents=True, exist_ok=True)

    # 1) 相関ヒートマップ（Pearson / Spearman）
    if "pearson" in dfc.columns:
        save_corr_heatmap(dfc, "pearson",
                          fig_dir / "Fig_bidirectional_corr_pearson.png",
                          fig_dir / "Fig_bidirectional_corr_pearson.pdf",
                          "Bidirectional correlations (Pearson)")
    if "spearman" in dfc.columns:
        save_corr_heatmap(dfc, "spearman",
                          fig_dir / "Fig_bidirectional_corr_spearman.png",
                          fig_dir / "Fig_bidirectional_corr_spearman.pdf",
                          "Bidirectional correlations (Spearman)")

    # 2) 代表散布図（順方向 ARI_mean vs 逆方向の代表指標） —— 列名は環境に合わせて調整
    #   よく使う候補名を順に探す（存在するものだけ描画）
    x_candidates = ["ARI_mean", "forward_score", "ARI"]  # 順方向側のスコア列候補
    y_candidates = [
        "reverse_score",
        "mean_OH_purity",
        "pair_OH_major_same_rate",
        "pair_mean_cosine_similarity",
        "FPcluster_median_cohesive_ratio",
    ]

    def pick_first_existing(df, cands):
        return [c for c in cands if c in df.columns]

    xs = pick_first_existing(dfj, x_candidates)
    ys = pick_first_existing(dfj, y_candidates)

    if xs and ys:
        x = xs[0]
        for y in ys:
            scatter_pair(
                dfj, x, y,
                fig_dir / f"Fig_scatter_{x}_vs_{y}.png",
                fig_dir / f"Fig_scatter_{x}_vs_{y}.pdf",
                f"{x} vs {y}"
            )

    print("\n✅ Done. Figures ->", fig_dir)

if __name__ == "__main__":
    main()


[INFO] correlations を joined から再計算します…
[SAVE] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/paper_4.2.4.4_OHFP/bidirectional_summary/Table_4.2.4.4_bidirectional_correlations.csv

✅ Done. Figures -> /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/paper_4.2.4.4_OHFP/bidirectional_summary/figs_from_joined


In [8]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
viz_bidirectional_4244.py
-------------------------
4.2.4.4 双方向（OH→FP と FP→OH）比較の可視化まとめ。

入力（既定パスに無い場合は --base で指定）:
  <base>/paper_4.2.4.4_OHFP/bidirectional_summary/
    ├─ Table_4.2.4.4_bidirectional_joined.csv
    └─ Table_4.2.4.4_bidirectional_correlations.csv

出力:
  <base>/paper_4.2.4.4_OHFP/bidirectional_summary/visuals/
    ├─ Fig_4.2.4.4_corr_heatmap.png/pdf
    ├─ Fig_4.2.4.4_scatter_{X}_vs_{Y}.png/pdf      （主要ペア）
    ├─ Fig_4.2.4.4_scatter_labeled_{X}_vs_{Y}.png/pdf（mode/indexラベル注釈）
    └─ Fig_4.2.4.4_pairs_grid_{X}_vs_{Y}.png/pdf   （条件ごとグリッド）

実行例:
  python viz_bidirectional_4244.py
  python viz_bidirectional_4244.py --base /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/paper_4.2.4.4_OHFP_20251022
"""

from __future__ import annotations
import argparse, os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # GUIなし環境向け
import matplotlib.pyplot as plt

DEFAULT_BASE = Path("/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/paper_4.2.4.4_OHFP_20251022")

def load_tables(base: Path):
    src_dir = base / "paper_4.2.4.4_OHFP_20251022" / "bidirectional_summary"
    joined = src_dir / "Table_4.2.4.4_bidirectional_joined.csv"
    corrs  = src_dir / "Table_4.2.4.4_bidirectional_correlations.csv"
    if not joined.exists():
        raise FileNotFoundError(f"joined が見つかりません: {joined}")
    if not corrs.exists():
        raise FileNotFoundError(f"correlations が見つかりません: {corrs}")
    dfj = pd.read_csv(joined)
    dfc = pd.read_csv(corrs)
    return dfj, dfc, src_dir

def ensure_outdir(src_dir: Path):
    out = src_dir / "visuals"
    out.mkdir(parents=True, exist_ok=True)
    return out

def corr_heatmap_from_table(dfc: pd.DataFrame, outdir: Path):
    # correlations.csv -> ヒートマップ（Pearson と Spearman を2枚）
    for stat_col, title_suffix in [("pearson_r","(Pearson r)"), ("spearman_rho","(Spearman ρ)")]:
        # 行: forward_metric, 列: reverse_metric
        fms = list(dict.fromkeys(dfc["forward_metric"].astype(str)))
        rms = list(dict.fromkeys(dfc["reverse_metric"].astype(str)))
        M = np.full((len(fms), len(rms)), np.nan, dtype=float)
        for i, fm in enumerate(fms):
            for j, rm in enumerate(rms):
                sub = dfc[(dfc["forward_metric"]==fm) & (dfc["reverse_metric"]==rm)]
                if len(sub):
                    M[i,j] = float(sub.iloc[0][stat_col])

        fig, ax = plt.subplots(figsize=(1.2*len(rms)+2, 1.0*len(fms)+2), dpi=300)
        im = ax.imshow(M, aspect="auto", interpolation="nearest")
        ax.set_xticks(np.arange(len(rms))); ax.set_xticklabels(rms, rotation=35, ha="right")
        ax.set_yticks(np.arange(len(fms))); ax.set_yticklabels(fms)
        ax.set_title(f"Fig 4.2.4.4: Forward vs Reverse correlations {title_suffix}")
        ax.grid(False)
        cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.set_label("Correlation")
        # 数値を上に軽く入れる（NaNはスキップ）
        for i in range(len(fms)):
            for j in range(len(rms)):
                if not np.isnan(M[i,j]):
                    ax.text(j, i, f"{M[i,j]:.2f}", ha="center", va="center", fontsize=8)
        fig.tight_layout()
        fig.savefig(outdir / f"Fig_4.2.4.4_corr_heatmap_{stat_col}.png", bbox_inches="tight")
        fig.savefig(outdir / f"Fig_4.2.4.4_corr_heatmap_{stat_col}.pdf", bbox_inches="tight")
        plt.close(fig)

def scatter_with_fit(df, xcol, ycol, outdir: Path, label_points=False, grid_by_condition=False):
    # 散布 + 近似直線
    sub = df[[xcol, ycol, "mode", "index"]].copy()
    x = pd.to_numeric(sub[xcol], errors="coerce")
    y = pd.to_numeric(sub[ycol], errors="coerce")
    m = x.notna() & y.notna()
    if m.sum() < 2:
        return
    x, y, sub = x[m].values, y[m].values, sub[m]

    if grid_by_condition:
        # mode×index の小 multiples
        keys = [(r.mode, r.index) for r in sub.itertuples()]
        uniq = sorted(set(keys))
        n = len(uniq)
        cols = min(3, max(1, int(np.ceil(np.sqrt(n)))))
        rows = int(np.ceil(n / cols))
        fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 3.8*rows), dpi=300)
        if rows*cols == 1:
            axes = np.array([[axes]])
        axes = axes.reshape(rows, cols)
        k = 0
        for i in range(rows):
            for j in range(cols):
                ax = axes[i,j]
                if k >= n:
                    ax.axis("off")
                    continue
                mkey = uniq[k]
                sel = (sub["mode"]==mkey[0]) & (sub["index"]==mkey[1])
                xx = pd.to_numeric(sub.loc[sel, xcol], errors="coerce")
                yy = pd.to_numeric(sub.loc[sel, ycol], errors="coerce")
                mm = xx.notna() & yy.notna()
                if mm.sum() >= 2:
                    ax.scatter(xx[mm].values, yy[mm].values, alpha=0.9)
                    # fit
                    coef = np.polyfit(xx[mm].values, yy[mm].values, 1)
                    xs = np.linspace(float(np.min(xx[mm].values)), float(np.max(xx[mm].values)), 100)
                    ys = np.polyval(coef, xs)
                    ax.plot(xs, ys)
                ax.set_title(f"{mkey[0]} × {mkey[1]}")
                ax.set_xlabel(xcol); ax.set_ylabel(ycol)
                ax.grid(True, linestyle="--", linewidth=0.4, alpha=0.6)
                k += 1
        fig.suptitle(f"Fig 4.2.4.4: {xcol} vs {ycol} — by (mode,index)")
        fig.tight_layout(rect=[0,0,1,0.97])
        fig.savefig(outdir / f"Fig_4.2.4.4_pairs_grid_{xcol}_vs_{ycol}.png", bbox_inches="tight")
        fig.savefig(outdir / f"Fig_4.2.4.4_pairs_grid_{xcol}_vs_{ycol}.pdf", bbox_inches="tight")
        plt.close(fig)
        return

    fig, ax = plt.subplots(figsize=(6,4), dpi=300)
    ax.scatter(x, y, alpha=0.9)
    # 近似直線
    coef = np.polyfit(x, y, 1)
    xs = np.linspace(float(np.min(x)), float(np.max(x)), 100)
    ys = np.polyval(coef, xs)
    ax.plot(xs, ys)
    ax.set_xlabel(xcol); ax.set_ylabel(ycol)
    ttl = f"Fig 4.2.4.4: {xcol} vs {ycol}"
    ax.set_title(ttl)
    ax.grid(True, linestyle="--", linewidth=0.4, alpha=0.6)

    if label_points:
        for r in sub.itertuples():
            ax.text(getattr(r, xcol), getattr(r, ycol), f"{r.mode}-{r.index}",
                    ha="left", va="bottom", fontsize=8)

    fig.tight_layout()
    base = f"Fig_4.2.4.4_scatter_{'labeled_' if label_points else ''}{xcol}_vs_{ycol}"
    fig.savefig(outdir / (base + ".png"), bbox_inches="tight")
    fig.savefig(outdir / (base + ".pdf"), bbox_inches="tight")
    plt.close(fig)

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--base", type=str, default=str(DEFAULT_BASE),
                    help="プロジェクトのベースフォルダ（既定は提示いただいたパス）")
    args, _ = ap.parse_known_args()  # Jupyterの -f 回避
    base = Path(args.base).resolve()

    dfj, dfc, src_dir = load_tables(base)
    outdir = ensure_outdir(src_dir)

    # 1) 相関ヒートマップ（Pearson & Spearman）
    corr_heatmap_from_table(dfc, outdir)

    # 2) 主要ペアの散布図（全体 + ラベル付き + 条件グリッド）
    pairs = [
        ("ARI_mean", "pair_OH_major_same_rate"),
        ("ARI_mean", "pair_mean_cosine_similarity"),
        ("ARI_mean", "FPcluster_median_cohesive_ratio"),
        ("ARI_mean", "mean_OH_purity"),
        ("ARI_mean", "mean_OH_entropy"),
    ]
    for xcol, ycol in pairs:
        if xcol in dfj.columns and ycol in dfj.columns:
            scatter_with_fit(dfj, xcol, ycol, outdir, label_points=False, grid_by_condition=False)
            scatter_with_fit(dfj, xcol, ycol, outdir, label_points=True,  grid_by_condition=False)
            scatter_with_fit(dfj, xcol, ycol, outdir, label_points=False, grid_by_condition=True)

    print("\n✅ Done. Figures ->", outdir)

if __name__ == "__main__":
    main()


FileNotFoundError: joined が見つかりません: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/paper_4.2.4.4_OHFP_20251022/paper_4.2.4.4_OHFP_20251022/bidirectional_summary/Table_4.2.4.4_bidirectional_joined.csv